In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U trl peft accelerate bitsandbytes transformers datasets TRL

In [ ]:
from datasets import load_dataset

# Load the official v1.5 dataset from the AI-MO organization
dataset = load_dataset("AI-MO/NuminaMath-1.5", split="train[:5000]")

# Look at how the fields are structured now
print(dataset[0].keys())

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-Math-1.5B-Instruct"

# Configure 4-bit loading to save VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" 
)

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters() # training < 2% of the model!

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# 1. Load and Format Dataset
print("Loading dataset...")
dataset = load_dataset("AI-MO/NuminaMath-1.5", split="train[:5000]")

def formatting_prompts_func(examples):
    output_text = []
    for p, s in zip(examples['problem'], examples['solution']):
        output_text.append(f"### Problem:\n{p}\n\n### Solution:\n{s}")
    return { "text" : output_text }

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

# 2. Load Base Model & Tokenizer
print("Loading model in 4-bit...")
model_id = "Qwen/Qwen2.5-Math-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# CRITICAL FIX: We load the raw base model and do NOT use get_peft_model()
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. Define LoRA Config
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Configure Training
training_args = SFTConfig(
    output_dir="./math_llm_output",
    dataset_text_field="text",       
    max_length=512,                  
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100, 
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

# 5. Initialize Trainer
print("Starting trainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args,
    peft_config=peft_config,    # <-- CRITICAL FIX: We pass the LoRA config here!
    processing_class=tokenizer      
)

# 6. Train!
trainer.train()

In [1]:
!pip install -U bitsandbytes

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# 1. Load Dataset
print("Loading dataset...")
dataset = load_dataset("AI-MO/NuminaMath-1.5", split="train[:5000]")

def formatting_prompts_func(examples):
    output_text = []
    for p, s in zip(examples['problem'], examples['solution']):
        output_text.append(f"### Problem:\n{p}\n\n### Solution:\n{s}")
    return { "text" : output_text }

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

# 2. Load Tokenizer
print("Loading tokenizer...")
model_id = "Qwen/Qwen2.5-Math-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 3. Tokenize Dataset (Required for standard Trainer)
print("Tokenizing dataset...")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
# Remove old string columns so the Trainer doesn't get confused
tokenized_dataset = tokenized_dataset.remove_columns([col for col in dataset.column_names] + ["text"])

# 4. Load Model
print("Loading 4-bit model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 5. Apply LoRA Manually
print("Applying LoRA...")
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 6. Use the Standard, Stable Trainer
print("Starting training loop...")
training_args = TrainingArguments(
    output_dir="./math_llm_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100, 
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator
)

trainer.train()

Loading dataset...
Loading tokenizer...
Tokenizing dataset...
Loading 4-bit model...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# Save LoRA adapter
# trainer.save_model("./math_llm_output/final")
# tokenizer.save_pretrained("./math_llm_output/final")

# Inference
from peft import PeftModel

prompt = "### Problem:\nWhat is the derivative of x^3 + 2x?\n\n### Solution:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

it degenerates into garbage table tokens — classic sign of undertrained model (only 100 steps, 0.16 epoch) hitting max_new_tokens=200 and rambling once it has nothing left to say.

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.3
)

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# 1. Load Dataset
print("Loading dataset...")
dataset = load_dataset("AI-MO/NuminaMath-1.5", split="train[:5000]")

# 2. Load Tokenizer
print("Loading tokenizer...")
model_id = "Qwen/Qwen2.5-Math-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 3. Format with EOS so model learns to stop
def formatting_prompts_func(examples):
    output_text = []
    for p, s in zip(examples['problem'], examples['solution']):
        output_text.append(f"### Problem:\n{p}\n\n### Solution:\n{s}{tokenizer.eos_token}")
    return { "text" : output_text }

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

# 4. Tokenize (no extra special tokens since we added EOS manually)
print("Tokenizing dataset...")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, add_special_tokens=False)

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns([col for col in dataset.column_names] + ["text"])

# 5. Load Model
print("Loading 4-bit model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 6. Apply LoRA
print("Applying LoRA...")
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 7. Train — more steps this time
print("Starting training loop...")
training_args = TrainingArguments(
    output_dir="./math_llm_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    max_steps=600,
    fp16=True,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=200,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator
)

trainer.train()

# 8. Save
trainer.save_model("./math_llm_output/final")
tokenizer.save_pretrained("./math_llm_output/final")
from google.colab import drive
drive.mount('/content/drive')

# after training
trainer.save_model("/content/drive/MyDrive/math_llm_output/final")
tokenizer.save_pretrained("/content/drive/MyDrive/math_llm_output/final")

Loading dataset...
Loading tokenizer...
Tokenizing dataset...
Loading 4-bit model...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Applying LoRA...
Starting training loop...


Step,Training Loss
20,1.331661
40,1.124945
60,1.097998
80,1.159140
100,1.147941
120,1.111945
140,1.037582
160,1.060121
180,1.029691
200,1.006073


NotImplementedError: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See examples at https://colab.research.google.com/notebooks/io.ipynb#scrollTo=7taylj9wpsA2.

In [5]:
import shutil
shutil.make_archive("math_llm_final", "zip", "./math_llm_output/final")

'/kaggle/working/math_llm_final.zip'

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_id = "Qwen/Qwen2.5-Math-1.5B-Instruct"
adapter_path = "./math_llm_output/final"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# Base model (same 4-bit config as training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Attach LoRA adapter
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [7]:
prompt = "### Problem:\nWhat is the derivative of x^3 + 2x?\n\n### Solution:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.3
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Problem:
What is the derivative of x^3 + 2x?

### Solution:
The answer to this question will be marked by the system, but we need not only provide it. It must also include a complete solution and justification.

Solution: The correct calculation result with an error in units or unit conversion - for example, $60$ km/h instead of $\frac{1}{5}$ m/s

Inadequate handling of algebraic operations (for instance, incorrect use of exponents) - for example, $(\cos(x))^{\prime} = \sin^{2}(x)$

Notation errors/numbering problems - for example, writing "4" as "7"

Mathematical notation used incorrectly / misunderstanding (e.g., using letters other than numbers when no specific values are needed)

If the problem statement includes any additional conditions, they should be clearly stated within the text; otherwise, their presence has been assumed. If these terms do appear without further explanation, then all necessary mathematical expressions can be provided directly after them. However, if ther

In [10]:
for _ in range(4):
    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False, temperature=0.1, top_p=0.9, pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id, repetition_penalty=1.3)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("---")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### Problem:
What is the derivative of x^3 + 2x?

### Solution:
Answer: $3 x^{2}+2$.

Solution. The sum rule for derivatives states that if a function is composed from several functions, then its derivative can be found by differentiating each component and adding them up; in other words,

$$
\frac{d}{dx}\left(f_{1}(x)+f_{2}(x)\right)=\frac{d f_{1}}{\mathrm{~d } x}(x)+\frac{d f_{2}}{\mathrm{~d } x}(x)
$$

The power rule for differentiation tells us how to differentiate powers:

$$
\begin{aligned}
& \text { If } y=x^{n}, n=0,1,2,3,4,5,6, \ldots ; \\
& \quad \frac{d y}{d x}=n x^{n-1}.
\end{aligned}
$$

Applying these rules, we get $\frac{d
---
### Problem:
What is the derivative of x^3 + 2x?

### Solution:
Answer: $3 x^{2}+2$.

Solution. The sum rule for derivatives states that if a function is composed from several functions, then its derivative can be found by differentiating each component and adding them up; in other words,

$$
\frac{d}{dx}\left(f_{1}(x)+f_{2}(x)\right)=\frac{d f_{1}